# U6-04 · Anomalías, reglas de asociación y segmentación

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 6, apartados 6.8 a 6.10*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

## 1. Lecturas anómalas de una máquina (ejemplo 8 y figura 6.6)

500 lecturas normales simuladas (la vibración sube con la temperatura) y 10 anomalías añadidas a mano.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
n = 500
temp = rng.normal(60, 2, n)
vib = 0.5 * (temp - 60) + rng.normal(4, 0.6, n)
an_t = np.array([61, 59, 66, 52, 63, 60, 70, 58, 65, 55])
an_v = np.array([9.5, 0.8, 4.0, 4.2, 8.0, 7.6, 9.0, 1.0, 2.0, 7.0])
X_sensor = np.column_stack([np.r_[temp, an_t], np.r_[vib, an_v]])
es_anomalia = np.r_[np.zeros(n), np.ones(10)].astype(bool)

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=0).fit(X_sensor)
marcadas = iso.predict(X_sensor) == -1
puntuacion = -iso.score_samples(X_sensor)
print("marcadas:", marcadas.sum(), "· anomalías detectadas:", (marcadas & es_anomalia).sum(),
      "de 10 · falsas alarmas:", (marcadas & ~es_anomalia).sum())
print("anomalías entre las 15 más altas:", es_anomalia[np.argsort(-puntuacion)[:15]].sum())
print("AUC de la puntuación:", round(roc_auc_score(es_anomalia, puntuacion), 3))
print("no detectadas:", X_sensor[es_anomalia & ~marcadas].tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*X_sensor[~es_anomalia & ~marcadas].T, s=6, c="lightgray", label="normal")
ax.scatter(*X_sensor[~es_anomalia & marcadas].T, s=25, facecolors="none", edgecolors="k", label="falsa alarma")
ax.scatter(*X_sensor[es_anomalia & marcadas].T, s=30, marker="D", label="anomalía detectada")
ax.scatter(*X_sensor[es_anomalia & ~marcadas].T, s=40, marker="x", label="anomalía no detectada")
ax.set_xlabel("temperatura (ºC)"); ax.set_ylabel("vibración (mm/s)"); ax.legend(fontsize=8)
plt.show()

**Prueba.** Cambia `contamination` a 0.05. ¿Cuántas anomalías detecta ahora y cuántas falsas alarmas da?

## 2. Recambios que se piden juntos (ejemplo 9)

Calculamos soporte, confianza y *lift* a mano, sin librerías adicionales.

In [ ]:
ordenes = [{"rodamiento", "grasa", "junta"}, {"rodamiento", "grasa"}, {"filtro", "aceite"},
           {"rodamiento", "grasa", "correa"}, {"filtro", "aceite", "junta"}, {"correa"},
           {"rodamiento", "junta"}, {"filtro", "aceite"}, {"rodamiento", "grasa"}, {"aceite"}]

def soporte(conjunto):
    return sum(conjunto <= o for o in ordenes) / len(ordenes)

def regla(a, b):
    s_ab = soporte({a, b})
    conf = s_ab / soporte({a})
    return {"regla": f"{a} => {b}", "soporte": s_ab, "confianza": round(conf, 3),
            "lift": round(conf / soporte({b}), 3)}

piezas = sorted(set().union(*ordenes))
reglas = pd.DataFrame([regla(a, b) for a in piezas for b in piezas if a != b and soporte({a, b}) > 0])
reglas.sort_values("lift", ascending=False)

**Prueba.** Filtra las reglas con soporte ≥ 0,2 y confianza ≥ 0,8. ¿Cuáles quedan?

## 3. Una segmentación de principio a fin (ejemplo 10)

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.metrics import adjusted_rand_score

X, y = load_wine(return_X_y=True, as_frame=True)
segmentador = make_pipeline(
    StandardScaler(), PCA(n_components=5),
    KMeans(n_clusters=3, n_init=10, random_state=0))
segmento = segmentador.fit_predict(X)
print("varianza conservada:", segmentador[1].explained_variance_ratio_.sum().round(3))
print("tamaños:", np.bincount(segmento), "· ARI frente a los cultivares:", round(adjusted_rand_score(y, segmento), 3))
perfil = X.groupby(segmento)[["alcohol", "flavanoids", "color_intensity", "proline"]].mean()
perfil.loc["media"] = X[["alcohol", "flavanoids", "color_intensity", "proline"]].mean()
perfil.round(2)

**Ejercicio.** Comprueba la estabilidad: repite la segmentación con `random_state` de 0 a 9 y calcula el ARI de cada resultado frente al primero.

> Liu, F. T., Ting, K. M. y Zhou, Z.-H. (2008). Isolation Forest. *Proc. ICDM 2008*, 413–422. https://doi.org/10.1109/ICDM.2008.17
>
> Agrawal, R., Imieliński, T. y Swami, A. (1993). Mining association rules between sets of items in large databases. *Proc. SIGMOD 1993*, 207–216. https://doi.org/10.1145/170035.170072